In [1]:
import sys
sys.path.append('/Users/rashe/PycharmProjects/PredictingNBACareers')
from utils.duckdb import get_duckdb_connection
import duckdb
import polars as pl

In [2]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
select 
  p.name as Player,
  psv."value" as Stat_Value,
  s."name" as Stat_Name,
  psv.season_id,
  psv.player_id,
  psv.team_id 
from nba_data.main.player_stat_values psv
join nba_data.main.stats s on psv.stat_id = s.stat_id 
join nba_data.main.stat_types st on psv.stat_type_id = st.stat_type_id 
join nba_data.main.players p on psv.player_id = p.player_id 
where st.stat_type_id in (2, 3);
        """
    ).arrow()
    df_player_stats = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [3]:
df_player_stats

Player,Stat_Value,Stat_Name,season_id,player_id,team_id
str,f32,str,i32,i32,i32
"""Dean Wade""",0.3,"""BPM""",22,2328,33
"""Trey Murphy III""",0.091,"""WS/48""",22,3119,4
"""Miles McBride""",12.6,"""PER""",22,3017,20
"""Ivica Zubac""",22.299999,"""PER""",22,265,17
"""Antonio Reeves""",0.6,"""BLK%""",22,3108,4
…,…,…,…,…,…
"""KZ Okpala""",0.1,"""OWS""",33,2321,13
"""Luke Kennard""",0.704,"""3PAr""",33,516,24
"""Max Strus""",0.5,"""BLK%""",33,2300,27


In [7]:
df_player_stats_pivoted = df_player_stats.pivot(
    on="Stat_Name",
    index=["Player", "season_id", "player_id", "team_id"],
    values="Stat_Value",
    aggregate_function="first"
)
df_player_stats_pivoted

Player,season_id,player_id,team_id,BPM,WS/48,PER,BLK%,GS,Player_Age,FTr,VORP,OWS,TOV%,ORB%,TRB%,WS,3PAr,STL%,OBPM,DWS,AST%,DBPM,MP,USG%,TS%,DRB%,G,FT,2P,2P%,eFG%,2PA,ORB,FTA,AST,Trp-Dbl,FT%,PTS,BLK,TOV,FG%,3PA,PF,DRB,FGA,FG,STL,3P%,3P,TRB
str,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""Dean Wade""",22,2328,33,0.3,0.113,10.1,1.4,30.0,28.0,0.111,0.7,1.3,7.2,4.5,10.8,3.0,0.819,1.6,-1.0,1.7,7.3,1.2,1252.0,10.4,0.563,16.6,59.0,16.0,32.0,0.653,0.561,49.0,51.0,30.0,76.0,0.0,0.533,320.0,20.0,22.0,0.413,222.0,101.0,198.0,271.0,112.0,42.0,0.36,80.0,249.0
"""Trey Murphy III""",22,3119,4,0.8,0.091,17.299999,2.0,51.0,24.0,0.274,1.3,3.0,9.9,2.7,7.8,3.5,0.522,1.5,2.1,0.5,15.9,-1.4,1854.0,23.700001,0.598,13.2,53.0,204.0,223.0,0.556,0.548,401.0,47.0,230.0,188.0,0.0,0.887,1124.0,39.0,103.0,0.454,438.0,112.0,221.0,839.0,381.0,56.0,0.361,158.0,268.0
"""Miles McBride""",22,3017,20,0.0,0.098,12.6,1.0,10.0,24.0,0.117,0.8,1.8,6.7,3.4,5.8,3.2,0.57,2.1,-0.4,1.4,15.6,0.4,1593.0,16.799999,0.531,8.1,64.0,52.0,107.0,0.455,0.511,235.0,47.0,64.0,187.0,0.0,0.813,611.0,17.0,41.0,0.406,312.0,108.0,114.0,547.0,222.0,67.0,0.369,115.0,161.0
"""Ivica Zubac""",22,265,17,3.1,0.215,22.299999,3.4,80.0,27.0,0.251,3.4,7.0,10.8,13.1,21.799999,11.7,0.0,1.0,2.3,4.7,12.8,0.8,2624.0,19.5,0.641,30.200001,80.0,156.0,592.0,0.628,0.628,942.0,300.0,236.0,214.0,1.0,0.661,1340.0,90.0,127.0,0.628,0.0,168.0,710.0,942.0,592.0,55.0,null,0.0,1010.0
"""Antonio Reeves""",22,3108,4,-3.2,0.029,11.7,0.6,6.0,24.0,0.141,-0.2,0.4,10.2,2.9,5.1,0.4,0.52,1.5,-1.2,0.0,8.8,-2.0,660.0,18.799999,0.579,7.6,44.0,28.0,62.0,0.521,0.558,119.0,18.0,35.0,39.0,0.0,0.8,305.0,4.0,30.0,0.456,129.0,40.0,45.0,248.0,113.0,21.0,0.395,51.0,63.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Milt Palacio""",15,1437,36,-4.4,-0.041,8.8,0.0,0.0,21.0,0.378,-0.2,-0.5,27.799999,5.1,7.7,-0.3,0.02,2.7,-4.0,0.2,19.700001,-0.4,394.0,18.299999,0.473,10.4,53.0,22.0,43.0,0.448,0.439,96.0,17.0,37.0,48.0,0.0,0.595,108.0,0.0,44.0,0.439,2.0,32.0,34.0,98.0,43.0,20.0,0.0,0.0,51.0
"""John Starks""",15,967,24,-1.9,0.026,12.4,0.2,30.0,34.0,0.109,0.0,0.4,10.6,0.8,4.5,0.6,0.315,1.8,-0.6,0.3,26.200001,-1.3,1190.0,22.6,0.453,8.6,37.0,50.0,144.0,0.388,0.429,371.0,10.0,59.0,181.0,0.0,0.847,515.0,4.0,67.0,0.375,171.0,102.0,91.0,542.0,203.0,42.0,0.345,59.0,101.0
"""Nazr Mohammed""",15,1548,32,-7.0,-0.016,7.0,4.6,3.0,22.0,0.407,-0.2,-0.4,22.0,9.2,14.5,-0.1,0.0,1.1,-6.8,0.3,1.6,-0.1,190.0,18.9,0.424,20.1,28.0,12.0,21.0,0.389,0.389,54.0,16.0,22.0,2.0,0.0,0.545,54.0,12.0,18.0,0.389,0.0,29.0,34.0,54.0,21.0,4.0,null,0.0,50.0


In [10]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
select 
  p.name as Player,
  po.name as Position,
  p.player_id
from nba_data.main.players p
join nba_data.main.player_positions pp on p.player_id = pp.player_id
join nba_data.main.positions po on pp.position_id = po.position_id
        """
    ).arrow()
    df_player_positions = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [12]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
select 
  t.team_id,
  t.name as Team_Name
from nba_data.main.teams t
        """
    ).arrow()
    df_teams = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [13]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
select 
  s.season_id,
  s.label as Season
from nba_data.main.seasons s
        """
    ).arrow()
    df_seasons = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [14]:
df_seasons

season_id,Season
i32,i32
1,2017
2,1983
3,1994
4,1978
5,2000
…,…
46,1995
47,2001
48,1989


In [ ]:
df_player_positions = df_player_positions.group_by("Player").agg(
    pl.col("Position").unique().sort().alias("Positions") 
).with_columns(
    pl.col("Positions").list.join("/").alias("Positions")
)

In [23]:
full_df = df_player_stats_pivoted.join(
    df_player_positions,
    on="Player",
    how="left"
).join(
    df_teams,
    on="team_id",
    how="left"
).join(
    df_seasons,
    on="season_id",
    how="left"
)
full_df

Player,season_id,player_id,team_id,BPM,WS/48,PER,BLK%,GS,Player_Age,FTr,VORP,OWS,TOV%,ORB%,TRB%,WS,3PAr,STL%,OBPM,DWS,AST%,DBPM,MP,USG%,TS%,DRB%,G,FT,2P,2P%,eFG%,2PA,ORB,FTA,AST,Trp-Dbl,FT%,PTS,BLK,TOV,FG%,3PA,PF,DRB,FGA,FG,STL,3P%,3P,TRB,Positions,Team_Name,Season
str,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str,str,i32
"""Dean Wade""",22,2328,33,0.3,0.113,10.1,1.4,30.0,28.0,0.111,0.7,1.3,7.2,4.5,10.8,3.0,0.819,1.6,-1.0,1.7,7.3,1.2,1252.0,10.4,0.563,16.6,59.0,16.0,32.0,0.653,0.561,49.0,51.0,30.0,76.0,0.0,0.533,320.0,20.0,22.0,0.413,222.0,101.0,198.0,271.0,112.0,42.0,0.36,80.0,249.0,"""PF""","""Cleveland Cavaliers""",2024
"""Trey Murphy III""",22,3119,4,0.8,0.091,17.299999,2.0,51.0,24.0,0.274,1.3,3.0,9.9,2.7,7.8,3.5,0.522,1.5,2.1,0.5,15.9,-1.4,1854.0,23.700001,0.598,13.2,53.0,204.0,223.0,0.556,0.548,401.0,47.0,230.0,188.0,0.0,0.887,1124.0,39.0,103.0,0.454,438.0,112.0,221.0,839.0,381.0,56.0,0.361,158.0,268.0,"""SF""","""New Orleans Pelicans""",2024
"""Miles McBride""",22,3017,20,0.0,0.098,12.6,1.0,10.0,24.0,0.117,0.8,1.8,6.7,3.4,5.8,3.2,0.57,2.1,-0.4,1.4,15.6,0.4,1593.0,16.799999,0.531,8.1,64.0,52.0,107.0,0.455,0.511,235.0,47.0,64.0,187.0,0.0,0.813,611.0,17.0,41.0,0.406,312.0,108.0,114.0,547.0,222.0,67.0,0.369,115.0,161.0,"""PG""","""New York Knicks""",2024
"""Ivica Zubac""",22,265,17,3.1,0.215,22.299999,3.4,80.0,27.0,0.251,3.4,7.0,10.8,13.1,21.799999,11.7,0.0,1.0,2.3,4.7,12.8,0.8,2624.0,19.5,0.641,30.200001,80.0,156.0,592.0,0.628,0.628,942.0,300.0,236.0,214.0,1.0,0.661,1340.0,90.0,127.0,0.628,0.0,168.0,710.0,942.0,592.0,55.0,null,0.0,1010.0,"""C""","""Los Angeles Clippers""",2024
"""Antonio Reeves""",22,3108,4,-3.2,0.029,11.7,0.6,6.0,24.0,0.141,-0.2,0.4,10.2,2.9,5.1,0.4,0.52,1.5,-1.2,0.0,8.8,-2.0,660.0,18.799999,0.579,7.6,44.0,28.0,62.0,0.521,0.558,119.0,18.0,35.0,39.0,0.0,0.8,305.0,4.0,30.0,0.456,129.0,40.0,45.0,248.0,113.0,21.0,0.395,51.0,63.0,"""SG""","""New Orleans Pelicans""",2024
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Milt Palacio""",15,1437,36,-4.4,-0.041,8.8,0.0,0.0,21.0,0.378,-0.2,-0.5,27.799999,5.1,7.7,-0.3,0.02,2.7,-4.0,0.2,19.700001,-0.4,394.0,18.299999,0.473,10.4,53.0,22.0,43.0,0.448,0.439,96.0,17.0,37.0,48.0,0.0,0.595,108.0,0.0,44.0,0.439,2.0,32.0,34.0,98.0,43.0,20.0,0.0,0.0,51.0,"""PG""","""Vancouver Grizzlies""",1999
"""John Starks""",15,967,24,-1.9,0.026,12.4,0.2,30.0,34.0,0.109,0.0,0.4,10.6,0.8,4.5,0.6,0.315,1.8,-0.6,0.3,26.200001,-1.3,1190.0,22.6,0.453,8.6,37.0,50.0,144.0,0.388,0.429,371.0,10.0,59.0,181.0,0.0,0.847,515.0,4.0,67.0,0.375,171.0,102.0,91.0,542.0,203.0,42.0,0.345,59.0,101.0,"""SG""","""Unknown""",1999
"""Nazr Mohammed""",15,1548,32,-7.0,-0.016,7.0,4.6,3.0,22.0,0.407,-0.2,-0.4,22.0,9.2,14.5,-0.1,0.0,1.1,-6.8,0.3,1.6,-0.1,190.0,18.9,0.424,20.1,28.0,12.0,21.0,0.389,0.389,54.0,16.0,22.0,2.0,0.0,0.545,54.0,12.0,18.0,0.389,0.0,29.0,34.0,54.0,21.0,4.0,null,0.0,50.0,"""C""","""Philadelphia 76ers""",1999


In [39]:
from pathlib import Path
dirPath = Path.cwd().parent.parent.joinpath("data", "raw_data", "NBA_Player_Data")
dirPath.mkdir(parents=True, exist_ok=True)
full_df.write_csv(dirPath / "full_player_data.csv", include_header=True, separator=",")

In [42]:
filtered_df = full_df.filter(pl.col("Season") > 1999)
filtered_df.write_csv(dirPath / "filtered_player_data_after_1999.csv", include_header=True, separator=",")